<a href="https://colab.research.google.com/github/Satya-Padala/24P31A42V7_DriveReady_OPC_2026/blob/main/Day-5/task5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import chromadb
import sentence_transformers
import langchain
import langgraph
import langchain_groq

In [10]:
from groq import Groq
from getpass import getpass

api_key = getpass("Enter Groq API Key: ")

client = Groq(api_key=api_key)

print("Groq connected successfully!")

Enter Groq API Key: ··········
Groq connected successfully!


In [51]:
business_facts = [
    "Business name: Santhi's Pickles.",
    "Business type: A pickle business that accepts customer orders.",
    "Available pickle varieties include chicken pickle.",
    "Available pickle varieties include prawn pickle.",
    "Available pickle varieties include mango pickle.",
    "Chicken pickle costs Rs 1000 per kg.",
    "Prawn pickle costs Rs 900 per kg.",
    "Mango pickle costs Rs 800 per kg.",
    "Customers can order different quantities of available pickles.",
    "The order total is calculated by multiplying the price per kg by the quantity in kg.",
    "Orders are prepared after the customer places an order.",
    "Preparing an order takes approximately 6 days.",
    "The shop is open Monday from 9 AM to 8 PM.",
    "The shop is open Tuesday from 9 AM to 8 PM.",
    "The shop is open Wednesday from 9 AM to 8 PM.",
    "The shop is open Thursday from 9 AM to 8 PM.",
    "The shop is open Friday from 9 AM to 8 PM.",
    "The shop is open Saturday from 9 AM to 8 PM.",
    "Sunday opening hours have not been specified; customers should be asked to confirm.",
    "Delivery areas, delivery charges, payment methods, and cancellation/refund policies have not been specified. The assistant must not guess them and should ask the business owner for confirmation."
]

print("Total business facts:", len(business_facts))

Total business facts: 20


In [5]:
import chromadb
from chromadb.utils import embedding_functions

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

chroma_client = chromadb.PersistentClient(
    path="./santhi_pickle_db"
)

collection = chroma_client.get_or_create_collection(
    name="santhi_pickles",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine"}
)

collection.upsert(
    ids=[f"fact_{i}" for i in range(len(business_facts))],
    documents=business_facts
)

print("Facts loaded successfully!")
print("Total stored facts:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Facts loaded successfully!
Total stored facts: 20


In [7]:
from langchain_core.tools import tool

@tool
def search_pickle_shop(question: str) -> str:
    """
    Search Santhi's Pickles business knowledge.

    Topics covered:
    - Business name and business type
    - Available pickle varieties
    - Chicken pickle price per kg
    - Prawn pickle price per kg
    - Mango pickle price per kg
    - Order quantities and total price calculation
    - Order preparation process
    - Order preparation time
    - Monday to Saturday opening hours
    - Sunday opening hours
    - Delivery areas and charges
    - Payment methods
    - Cancellation and refund policies
    """

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    documents = results["documents"][0]
    distances = results["distances"][0]

    relevant_facts = []

    for doc, distance in zip(documents, distances):
        if distance <= 0.65:
            relevant_facts.append(doc)

    if not relevant_facts:
        return "NO RELEVANT BUSINESS INFORMATION FOUND."

    return "\n".join(relevant_facts)


print(
    search_pickle_shop.invoke(
        "What is the price of chicken pickle?"
    )
)
print(search_pickle_shop.invoke(
    "What are your shop opening hours on Monday?"
))

print(search_pickle_shop.invoke(
    "How much does 2 kg of prawn pickle cost?"
))

print(search_pickle_shop.invoke(
    "Do you provide home delivery?"
))

Chicken pickle costs Rs 1000 per kg.
Available pickle varieties include chicken pickle.
Prawn pickle costs Rs 900 per kg.
The shop is open Monday from 9 AM to 8 PM.
The shop is open Tuesday from 9 AM to 8 PM.
The shop is open Saturday from 9 AM to 8 PM.
Prawn pickle costs Rs 900 per kg.
Available pickle varieties include prawn pickle.
Mango pickle costs Rs 800 per kg.
Delivery areas, delivery charges, payment methods, and cancellation/refund policies have not been specified. The assistant must not guess them and should ask the business owner for confirmation.
Orders are prepared after the customer places an order.


In [23]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [48]:
SYSTEM = """
ROLE:
You are the friendly customer support assistant for Santhi's Pickles.

FACTS:
- Chicken pickle costs Rs 1000 per kg.
- Prawn pickle costs Rs 900 per kg.
- Mango pickle costs Rs 800 per kg.
- Orders are prepared after receiving them.
- Preparation takes approximately 6 days.
- Shop timings are Monday to Saturday, 9 AM to 8 PM.
- Sunday opening hours are not specified.
- Delivery areas, charges, payment methods and cancellation/refund
  policies are not specified.

RULES:
1. Always use get_price for prices.
2. Always use is_open for opening hours.
3. Use search_pickle_shop for business information.
4. Never invent business information.
5. If information is unavailable, say you will check with the owner.
6. Before log_order, collect name, item, quantity and pickup date.
7. Ask for any missing order details.
8. Calculate totals using price per kg multiplied by quantity.
9. Never assume Sunday opening or delivery policies.

FORMAT:
Reply politely in the customer's language.
Keep answers under 50 words.
"""

In [32]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

MENU = {"chicken pickle": 1000, "mango pickle": 800, "prawn pickle": 900}
HOURS = {"monday": "9 am to 8 pm", "tuesday": "9 am to 8 pm", "wednesday": "9 am to 8 pm",
         "thursday": "9 am to 8 pm", "friday": "9 am to 8 pm", "saturday": "9 am to 8 pm"}

class Order(BaseModel):
    """A pickle order taken from a customer's chat message."""
    name: str = Field(description="Customer's first name")
    item: Literal["chicken pickle", "mango pickle","prawn pickle"] = Field(description="Menu item")
    qty: int = Field(description="How many", ge=1)
    duration: int = Field(description="in how many days it should be delivered asked by customer")

In [33]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """does stuff
    """
    price = MENU.get(item.lower().strip())
    if price is None:
        return f"'{item}' is not on the menu. We have: {', '.join(MENU)}."
    return f"{qty} x {item} = Rs {price * qty}"

@tool
def is_open(day: str) -> str:
    """Check the shop opening hours for a day of the week, e.g. "sunday"."""
    hours = HOURS.get(day.lower().strip())
    return f"Open {hours} on {day}." if hours else f"'{day}' is not a day of the week."

@tool
def log_order(name: str, item: str, qty: int, duration: int) -> str:
    """Save a confirmed pickle order to the Orders sheet.
    Use only when the customer has given their name, item, quantity,duration"""
    try:
        order = Order(name=name, item=item.lower().strip(), qty=qty, duration=duration)
    except Exception as e:
        return f"Could not save the order: {e}. Ask the customer to correct it."
    total = MENU[order.item] * order.qty
    ws.append_row([order.name, order.item, order.qty, order.duration,total])
    return f"Saved: {order.qty} x {order.item} for {order.name} on {order.duration}, Rs {total}."

In [34]:
agent = create_agent(
    model=llm,
    tools=[
        get_price,
        is_open,
        log_order,
        search_pickle_shop
    ],
    system_prompt=SYSTEM,
    checkpointer=checkpointer
)

In [35]:
def chat(text: str, thread_id: str = "customer_1") -> str:

    try:
        result = agent.invoke(
            {"messages": [
                {"role": "user", "content": text}
            ]},
            config={
                "configurable": {
                    "thread_id": thread_id
                }
            }
        )

        return result["messages"][-1].content

    except Exception as e:
        print("Agent error:", e)
        return "Sorry, I'm a little busy. Please try again."

In [36]:
print(chat("My name is Satya. I want chicken pickle.", thread_id="customer_1"))

print(chat("How much will 2 kg cost?", thread_id="customer_1"))

print(chat("What is my name and which pickle did I select?", thread_id="customer_1"))

Satya, 2 kg of chicken pickle will be Rs 2000.  

When would you like to pick it up (after the 5‑6 day preparation)? Once I have the date, I’ll confirm and log your order.
2 kg of chicken pickle costs Rs 2000. Let me know your preferred pickup date so I can finalize the order.
Your name is Satya, and you selected chicken pickle.


In [37]:
print(chat("My name is Chinni. I want mango pickle.", thread_id="customer_2"))

print(chat("How much will 3 kg cost?", thread_id="customer_2"))

print(chat("What is my name and which pickle did I select?", thread_id="customer_2"))

Nice to meet you, Chinni! 🌟  

How many kilograms of mango pickle would you like, and which day would you like to pick it up?
Sure! Let me check the price for you.

**Mango pickle:** Rs 800 per kg  

**3 kg total:** Rs 2,400  

Could you let me know which day you’d like to pick up your order?
Your name is **Chinni**, and you selected **mango pickle**.


In [38]:
print("CUSTOMER 1:")
print(chat("What is my name and which pickle did I select?", thread_id="customer_1"))

print("\nCUSTOMER 2:")
print(chat("What is my name and which pickle did I select?", thread_id="customer_2"))

CUSTOMER 1:
Your name is Satya, and you chose chicken pickle.

CUSTOMER 2:
Your name is **Chinni**, and you chose **mango pickle**.


In [52]:
test_questions = [
    "Do you deliver to Hyderabad?",
    "What are your delivery charges?",
    "Which payment methods do you accept?",
    "Can I cancel my order and get a refund?",
    "Are you open on Sunday?"
]

for question in test_questions:
    print("\nQuestion:", question)
    print("Answer:", chat(question, thread_id="break_test"))
    print("-" * 60)


Question: Do you deliver to Hyderabad?
Answer: I’m not sure if we deliver to Hyderabad. I’ll check with the owner and let you know shortly.
------------------------------------------------------------

Question: What are your delivery charges?
Answer: I don’t have the exact delivery‑charge details right now. I’ll confirm with the owner and get back to you shortly.
------------------------------------------------------------

Question: Which payment methods do you accept?
Answer: I’m not sure about delivery to Hyderabad, the delivery fees, or the payment methods we accept. I’ll confirm these details with the owner and get back to you shortly.
------------------------------------------------------------

Question: Can I cancel my order and get a refund?
Answer: I don’t have the exact cancellation or refund policy details right now. I’ll verify with the owner and let you know shortly.
------------------------------------------------------------

Question: Are you open on Sunday?
Answer: 

In [56]:
tests = [
    ["What is the price of chicken pickle?", "1000"],
    ["How much does prawn pickle cost?", "900"],
    ["What is the price of mango pickle?", "800"],
    ["Do you sell vegetarian pickles?", "vegetarian"],
    ["Do you have non-vegetarian pickles?", "non-vegetarian"],
    ["How many days does preparation take?", "6 days"],
    ["Do you deliver on Sundays?", "Sunday"],
    ["When do you prepare the pickles?", "after receiving an order"],
    ["Do you deliver to Kakinada?", "check with the owner"],
    ["Are chicken and prawn pickles available?", "chicken"]
]

In [57]:
results = []

for i, (question, expected) in enumerate(tests):
    answer = chat(question, thread_id=f"final_test_{i}")

    passed = expected.lower() in answer.lower()

    results.append(passed)

    print(f"\nTest {i+1}: {'PASS' if passed else 'FAIL'}")
    print("Question:", question)
    print("Answer:", answer)
    print("-" * 50)


Test 1: PASS
Question: What is the price of chicken pickle?
Answer: Chicken pickle costs **Rs 1000 per kg**. Let me know if you’d like to order or need any other information!
--------------------------------------------------

Test 2: PASS
Question: How much does prawn pickle cost?
Answer: Prawn pickle costs **Rs 900 per kg**. Let me know if you’d like to place an order!
--------------------------------------------------

Test 3: PASS
Question: What is the price of mango pickle?
Answer: Mango pickle costs Rs 800 per kg.
--------------------------------------------------

Test 4: PASS
Question: Do you sell vegetarian pickles?
Answer: We don’t have vegetarian pickles listed currently. I’ll check with the owner and let you know shortly.
--------------------------------------------------

Test 5: FAIL
Question: Do you have non-vegetarian pickles?
Answer: Yes—Chicken pickle (₹1000 /kg) and Prawn pickle (₹900 /kg). Would you like to know more or place an order?
-----------------------------

In [55]:
total = len(results)
passed = sum(results)
failed = total - passed

pass_rate = (passed / total) * 100

print("Total Tests:", total)
print("Passed:", passed)
print("Failed:", failed)
print("Pass Rate:", round(pass_rate, 2), "%")

Total Tests: 10
Passed: 6
Failed: 4
Pass Rate: 60.0 %
